In [5]:
import numpy as np
import torch
from transformers import TimeSeriesTransformerConfig, TimeSeriesTransformerForPrediction

# -------------------------
# 1. Example data
# -------------------------
time_steps = 200
t = np.arange(time_steps)
series = 0.05 * t + np.sin(0.2 * t) + 0.2 * np.random.randn(time_steps)

data = torch.tensor(series, dtype=torch.float32)

# -------------------------
# 2. Config
# -------------------------
context_length = 48
prediction_length = 24

config = TimeSeriesTransformerConfig(
    prediction_length=prediction_length,
    context_length=context_length,
    lags_sequence=[1, 2, 3, 4, 5, 6, 7],
    num_time_features=1,
    num_dynamic_real_features=0,
    num_static_real_features=0,
    num_static_categorical_features=0,
)

model = TimeSeriesTransformerForPrediction(config)

# -------------------------
# 3. Prepare inputs correctly
# -------------------------
max_lag = max(config.lags_sequence)
past_length = config.context_length + max_lag

required_total_length = past_length + config.prediction_length
if len(data) < required_total_length:
    raise ValueError(
        f"Not enough data: need at least {required_total_length} points, got {len(data)}"
    )

past_values = data[:past_length].unsqueeze(0)  # shape: (batch_size=1, past_length)
future_values = data[past_length:past_length + prediction_length].unsqueeze(0)  # shape: (1, prediction_length)

past_time_features = torch.arange(past_length, dtype=torch.float32).view(1, past_length, 1)
future_time_features = torch.arange(
    past_length, past_length + prediction_length, dtype=torch.float32
).view(1, prediction_length, 1)

past_observed_mask = torch.ones_like(past_values)
future_observed_mask = torch.ones_like(future_values)

# -------------------------
# 4. Sanity checks
# -------------------------
print("past_values shape:", past_values.shape)
print("past_time_features shape:", past_time_features.shape)
print("past_observed_mask shape:", past_observed_mask.shape)
print("future_values shape:", future_values.shape)
print("future_time_features shape:", future_time_features.shape)
print("required past length:", past_length)

# -------------------------
# 5. Forward pass (training)
# -------------------------
outputs = model(
    past_values=past_values,
    past_time_features=past_time_features,
    past_observed_mask=past_observed_mask,
    future_values=future_values,
    future_time_features=future_time_features,
    future_observed_mask=future_observed_mask,
)

loss = outputs.loss
print("Training loss:", loss.item())

forecast = model.generate(
    past_values=past_values,
    past_time_features=past_time_features,
    past_observed_mask=past_observed_mask,
    future_time_features=future_time_features,
)

prediction = forecast.sequences.mean(dim=1)

print("Forecast:")
print(prediction)

Forecast:
tensor([[ 0.0524, -0.6059, -0.0330, -0.7018, -0.4791, -0.0567, -0.0360,  0.0794,
          0.2226, -0.1042, -0.4728,  0.1720,  0.1760, -0.1421, -0.0635, -0.2086,
         -0.2190, -0.0471, -0.1419, -0.2659, -0.2436, -0.1608, -0.2921, -0.2180]])
